# Piloto da Fase 3.7: origem de fotografias reais

Este notebook verifica se o **Open Images V7** pode fornecer uma amostra interna de fotografias com perfil de autor rastre?vel. Ele baixa o CSV oficial completo e apenas tr?s imagens para `/content`. N?o cria divis?es, n?o inicia treinamento, n?o acessa benchmarks externos e n?o altera os checkpoints existentes. A origem declarada n?o certifica que uma imagem n?o tenha sido editada por IA.

Execute as c?lulas de prepara??o em ordem e, depois, a c?lula marcada **PILOTO MANUAL**. O piloto funciona em CPU; n?o consuma quota de GPU nesta etapa. Os dados locais de `/content` desaparecem ap?s desconex?o.

In [ ]:
# 1. Depend?ncias do piloto; GPU n?o ? necess?ria.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'boto3', 'pillow'])
print('Depend?ncias do piloto dispon?veis.', flush=True)

In [ ]:
# 2. Monte o Drive apenas para ler a credencial GitHub j? cifrada.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
assert (ROOT / 'credenciais/github.enc').is_file(), 'Credencial GitHub n?o encontrada no Drive'
print('Drive montado; /content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase37-Pilot-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_7_openimages_pilot.py',)
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro o piloto da Fase 3.7 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Configure logs em tempo real e aviso peri?dico para o piloto.
import os, queue, subprocess, sys, threading, time
PILOT = CODE_ROOT / 'experimentos/fase3_7_openimages_pilot.py'
assert PILOT.is_file(), f'Script ausente: {PILOT}'
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo h? {(time.monotonic()-begin)/60:.1f} min; sem nova sa?da', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (c?digo {code}); veja o log acima')
    print(f'{label}: conclu?do em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('C?digo sincronizado:', CODE_ROOT)

## Piloto manual

A c?lula a seguir baixa o CSV de metadados integralmente, conta candidatos com autor e licen?a informados e testa o acesso an?nimo a tr?s JPEGs escolhidos de forma determin?stica. Pode demorar por causa do CSV; acompanhe os indicadores em MB e linhas. **N?o execute c?lulas de treino de outras fases para este piloto.**

In [ ]:
# 6. PILOTO MANUAL: n?o congela sele??o e n?o treina.
run_live('piloto Open Images 3.7', [sys.executable, '-u', str(PILOT)])

In [ ]:
# 7. Leitura do resumo local, sem download adicional.
import json
report = Path('/content/fase3_7_openimages_pilot/pilot_summary.json')
if not report.is_file():
    raise FileNotFoundError('Execute primeiro a c?lula 6 do piloto')
summary = json.loads(report.read_text(encoding='utf-8'))
print('CSV SHA-256:', summary['metadata_sha256'])
print('Candidatas:', summary['eligible_images'], '| autores distintos:', summary['distinct_authors'])
print('JPEGs testados:', len(summary['sample_images']))
for row in summary['sample_images']:
    print(row['image_id'], row['width'], 'x', row['height'], '|', row['bytes'], 'bytes')

## Depois do piloto

Guarde a sa?da da c?lula 6 e as contagens da c?lula 7. S? depois de verificar quantidade, acesso real aos arquivos, espa?o e licen?a ser? fixada uma sele??o interna no Drive e criada uma variante de treino isolada. Nenhuma imagem deste piloto deve entrar em teste externo ou ser usada para ajustar limiares.